<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-03-prompting/lesson-3.3-model-routing/practice/GCP_Capstone_3.3_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 3.3 — Chain-of-Thought & Model Routing

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Run this first. Installs the unified `google-genai` SDK, authenticates with Application Default Credentials (no API keys), and initializes a Vertex AI client. Change `PROJECT_ID` to your project.

Every exercise below depends on the `client`, `types`, and helpers defined here.

In [ ]:
%%bash
pip install -q google-genai pydantic scipy

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS

from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import Literal
import json, time

# Vertex AI client (unified google-genai SDK). enterprise=True routes through Vertex.
client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')

USD_INR = 85  # for any INR cost display
print('Client ready:', PROJECT_ID)

## Exercise 1: Thinking Token Comparison

**Difficulty:** Easy

Ask a math question with budget=0, 1024, 4096, 8192. Compare thinking tokens and accuracy.

1. Use the same math question for all budgets
2. Print thinking tokens and answer for each
3. Observe: more thinking = more tokens = potentially better accuracy

In [ ]:
question = 'What is 17 * 23 + 456 - 89?'

for budget in [0, 1024, 4096, 8192]:
    r = client.models.generate_content(
        model='gemini-3.6-flash', contents=question,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=budget),
            max_output_tokens=1000))
    think = r.usage_metadata.thoughts_token_count or 0
    out = r.usage_metadata.candidates_token_count or 0
    print(f'  budget={budget:<6} think={think:<5} out={out:<5} answer={r.text.strip()[:40]}')

## Exercise 2: Thought Inspection

**Difficulty:** Easy

Enable include_thoughts=True. Separate thought parts from answer parts for 3 queries.

1. Set include_thoughts=True in ThinkingConfig
2. Iterate response.candidates[0].content.parts
3. Check part.thought boolean

In [ ]:
def inspect_thoughts(prompt):
    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(
                thinking_budget=2048, include_thoughts=True)),
    )
    for part in response.candidates[0].content.parts:
        if not part.text: continue
        label = 'THOUGHT' if part.thought else 'ANSWER'
        print(f'{label}: {part.text[:100]}')

    meta = response.usage_metadata
    print(f'Input: {meta.prompt_token_count} | Output: {meta.candidates_token_count} | Thinking: {meta.thoughts_token_count}')

for q in ['What is 17 * 23 + 45 * 12?',
          'If a shirt costs 800 rupees after a 20% discount, what was the original price?',
          'A tank fills in 6 hours with one pipe and 4 hours with another. Both open?']:
    print(f'\n=== {q} ===')
    inspect_thoughts(q)

## Exercise 3: Complexity Classifier

**Difficulty:** Easy

Build classify_complexity() using Flash-Lite + text/x.enum. Test on 10 queries.

1. Define SIMPLE/MEDIUM/COMPLEX criteria
2. Use Flash-Lite with thinking_budget=0
3. Verify classification accuracy on known queries

In [ ]:
def classify_complexity(query):
    r = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=f"""Classify complexity:
SIMPLE: Factual lookups, greetings, definitions, classification
MEDIUM: Explanations, comparisons, summaries, standard code
COMPLEX: Multi-step math, architecture, debugging, proofs

Query: {query}""",
        config=types.GenerateContentConfig(
            response_mime_type='text/x.enum',
            response_schema={'type':'STRING','enum':['SIMPLE','MEDIUM','COMPLEX']},
            thinking_config=types.ThinkingConfig(thinking_budget=0)))
    return r.text

test_queries = [
    'What is Python?',
    'Compare REST vs GraphQL for microservices',
    'Prove sqrt(2) is irrational',
    'What time is it?',
    'Design a distributed cache with consistency guarantees',
    'Hello!',
    'Summarize the CAP theorem',
    'Debug a race condition in a multithreaded queue',
    'Define idempotency',
    'Write a Python function to reverse a linked list',
]
for q in test_queries:
    print(f'  {classify_complexity(q):<8} | {q}')

## Exercise 4: Structured CoT Schema

**Difficulty:** Medium

Build ReasonedAnswer with reasoning before answer. Test on 5 questions.

1. Define Pydantic model with reasoning, answer, confidence fields
2. Field ORDER matters: reasoning must come first
3. Compare answers with vs without reasoning field

In [ ]:
class ReasonedAnswer(BaseModel):
    reasoning: str = Field(description='Step-by-step reasoning')
    answer: str = Field(description='Final answer')
    confidence: Literal['high','medium','low']

def reasoned(question):
    r = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=question,
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=ReasonedAnswer,
            temperature=0.1,
            thinking_config=types.ThinkingConfig(thinking_budget=0)))
    return r.parsed

questions = [
    'A train travels 120km in 1.5 hours. Average speed?',
    'If 3 painters paint 3 walls in 3 hours, how long for 9 painters to paint 9 walls?',
    'What is 15% of 240?',
    'A rectangle has perimeter 20 and area 24. What are its sides?',
    'How many days are between Jan 1 and Mar 1 in a non-leap year?',
]
for q in questions:
    result = reasoned(q)
    print(f'Q: {q}')
    print(f'  Reasoning: {result.reasoning[:120]}')
    print(f'  Answer: {result.answer}  (confidence={result.confidence})\n')

## Exercise 5: Cost Calculator

**Difficulty:** Medium

Build cost_per_1k() function. Compare routed (70/25/5) vs always-Pro vs always-Flash.

1. Use actual Gemini pricing (Flash-Lite / Flash / Pro per-1M rates)
2. Include thinking token costs
3. Calculate percentage savings

In [ ]:
# Standard per-1M-token rates (input / output), Gemini 3.x
PRICING = {
    'gemini-3.1-flash-lite': {'input':0.25, 'output':1.50},
    'gemini-3.6-flash':      {'input':1.50, 'output':7.50},
    'gemini-3.1-pro-preview':{'input':2.00, 'output':12.00},
}

def cost_per_1k(dist):
    total = 0
    for tier, d in dist.items():
        n = d['queries']
        p = PRICING[d['model']]
        # thinking tokens are billed at the output rate
        c = n*d['avg_in']/1e6*p['input'] + n*(d['avg_out']+d['avg_think'])/1e6*p['output']
        print(f'  {tier:<10} {n:>4}q | {d["model"]:<25} | ${c:.3f}')
        total += c
    print(f'  TOTAL: ${total:.3f}  (Rs {total*USD_INR:.0f})')
    return total

print('ROUTED (70/25/5):')
routed = cost_per_1k({
    'simple':  {'queries':700,'model':'gemini-3.1-flash-lite','avg_in':200,'avg_out':100,'avg_think':0},
    'medium':  {'queries':250,'model':'gemini-3.6-flash','avg_in':400,'avg_out':250,'avg_think':1024},
    'complex': {'queries':50, 'model':'gemini-3.1-pro-preview','avg_in':600,'avg_out':500,'avg_think':8192},
})
print('\nALWAYS-PRO:')
pro = cost_per_1k({
    'all': {'queries':1000,'model':'gemini-3.1-pro-preview','avg_in':300,'avg_out':200,'avg_think':4000},
})
print('\nALWAYS-FLASH:')
flash = cost_per_1k({
    'all': {'queries':1000,'model':'gemini-3.6-flash','avg_in':300,'avg_out':200,'avg_think':1024},
})
print(f'\nSAVINGS: {(1-routed/pro)*100:.1f}% vs always-Pro | {(1-routed/flash)*100:.1f}% vs always-Flash')

## Exercise 6: Full Router Pipeline

**Difficulty:** Medium

Build route_and_generate(). Test with 10 queries, verify correct model selection per complexity.

1. Build routing table (SIMPLE/MEDIUM/COMPLEX)
2. Classify, then generate with routed config
3. Return response + metadata (model, thinking tokens, complexity)

In [ ]:
ROUTING_TABLE = {
    'SIMPLE':  {'model':'gemini-3.1-flash-lite', 'budget':0,    'temp':0.1, 'max':512},
    'MEDIUM':  {'model':'gemini-3.6-flash',      'budget':1024, 'temp':0.3, 'max':2048},
    'COMPLEX': {'model':'gemini-3.1-pro-preview','budget':8192, 'temp':0.7, 'max':8192},
}

def route_and_generate(query, system_instruction=''):
    complexity = classify_complexity(query)
    cfg = ROUTING_TABLE[complexity]
    r = client.models.generate_content(
        model=cfg['model'], contents=query,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=cfg['temp'], max_output_tokens=cfg['max'],
            thinking_config=types.ThinkingConfig(thinking_budget=cfg['budget'])))
    meta = r.usage_metadata
    return {'text':r.text, 'complexity':complexity, 'model':cfg['model'],
            'think_tokens':meta.thoughts_token_count or 0, 'out_tokens':meta.candidates_token_count}

for q in test_queries:
    result = route_and_generate(q)
    print(f'{result["complexity"]:<8} -> {result["model"]:<25} think={result["think_tokens"]:<5} | {q[:45]}')

## Exercise 7: Cascade Router

**Difficulty:** Challenge

Build cascade_route() that tries the cheapest model first and escalates only when it isn't confident. Gemini 3.x on Vertex does NOT expose token logprobs, so use the model's self-reported confidence via structured output.

1. Try Flash-Lite; ask for a `{answer, confidence}` JSON, keep it if `confidence >= 0.75`
2. Escalate to Flash if below threshold
3. Final fallback to Pro
4. Track the escalation rate across the queries

In [ ]:
from pydantic import BaseModel

# Cascade router: try the cheapest model first, escalate only when it isn't confident.
# NOTE: token logprobs (the ideal confidence signal) are NOT exposed for Gemini 3.x on
# Vertex -- response_logprobs raises "Logprobs is not supported for this model"; they
# exist only on older 2.0-era models. So we cascade on the model's SELF-REPORTED
# confidence via structured output (a practical proxy -- calibrate thresholds on real data).

class ConfidentAnswer(BaseModel):
    answer: str
    confidence: float  # 0.0-1.0: how sure the answer is correct AND complete

def cascade_route(query):
    stages = [
        ('gemini-3.1-flash-lite', 0),
        ('gemini-3.6-flash', 2048),
        ('gemini-3.1-pro-preview', 8192),
    ]
    thresholds = [0.75, 0.60]  # escalate if self-confidence is below this
    for i, (model, budget) in enumerate(stages):
        r = client.models.generate_content(
            model=model,
            contents='Answer the query, then set confidence (0-1) to how sure you '
                     'are your answer is correct and complete. Query: ' + query,
            config=types.GenerateContentConfig(
                thinking_config=types.ThinkingConfig(thinking_budget=budget),
                response_mime_type='application/json',
                response_schema=ConfidentAnswer))
        a = r.parsed
        if i == len(stages) - 1:
            return a.answer, model, 'last_resort'
        if a.confidence >= thresholds[i]:
            return a.answer, model, 'confident (%.2f)' % a.confidence
        print('  Escalating from %s (confidence=%.2f)' % (model, a.confidence))
    return a.answer, model, 'escalated'

escalations = 0
queries = ['What is Python?', 'Hello!', 'Define idempotency',
           'Prove sqrt(2) is irrational', 'Design a distributed cache with consistency guarantees']
for q in queries:
    text, model, reason = cascade_route(q)
    if model != 'gemini-3.1-flash-lite':
        escalations += 1
    print('%s (%s): %s...' % (model, reason, text[:60]))
print('Escalation rate: %d/%d' % (escalations, len(queries)))

## Exercise 8: model_router.py Module

**Difficulty:** Challenge

Build complete module integrating classify(), route(), structured output, and cost tracking.

1. classify() with Flash-Lite + text/x.enum
2. route() with routing table + optional schema
3. Test: route(query, schema=RAGAnswer) returns structured JSON

In [ ]:
# Production module: reuses ROUTING_TABLE from Exercise 6.
def classify(query):
    r = client.models.generate_content(
        model='gemini-3.1-flash-lite', contents=f'Classify: {query}',
        config=types.GenerateContentConfig(
            response_mime_type='text/x.enum',
            response_schema={'type':'STRING','enum':['SIMPLE','MEDIUM','COMPLEX']},
            thinking_config=types.ThinkingConfig(thinking_budget=0)))
    return r.text

def route(query, system_instruction='', schema=None):
    complexity = classify(query)
    cfg = ROUTING_TABLE[complexity]
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=cfg['temp'], max_output_tokens=cfg['max'],
        thinking_config=types.ThinkingConfig(thinking_budget=cfg['budget']))
    if schema:
        config.response_mime_type = 'application/json'
        config.response_schema = schema
    r = client.models.generate_content(model=cfg['model'], contents=query, config=config)
    return {'text':r.text, 'complexity':complexity, 'model':cfg['model'],
            'think_tokens':r.usage_metadata.thoughts_token_count or 0}

print('Module ready: classify(), route()')

# Structured-output integration test
class RAGAnswer(BaseModel):
    answer: str = Field(description='Grounded answer')
    sources: list[str] = Field(description='Source references')
    confidence: Literal['high','medium','low']

out = route('Explain what a vector database is and where you would use one.', schema=RAGAnswer)
print(f'complexity={out["complexity"]} model={out["model"]}')
print(json.dumps(json.loads(out['text']), indent=2)[:400])